In [6]:
"""
Coarsened Exact Matching (CEM): Chromebook Effect on GPA and Absence Rate
==========================================================================

Strategy:
  - Exact match on: sex, SPED status, multilingual status, race
  - Coarse match on: GradYr (binned), prior absence rate (binned)
  - After matching, estimate treatment effect on matched sample
  - Also run weighted regression on matched sample for robustness

All restricted to free/reduced lunch students.
"""

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

def ols_with_se(X, y, var_names=None, weights=None):
    """WLS with HC1 robust SEs. If weights=None, does OLS."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, k = X.shape
    if weights is not None:
        w = np.asarray(weights, dtype=float)
        W_sqrt = np.sqrt(w)
        Xw = X * W_sqrt[:, None]
        yw = y * W_sqrt
    else:
        Xw, yw = X, y
        w = np.ones(n)

    beta = np.linalg.lstsq(Xw, yw, rcond=None)[0]
    resid = y - X @ beta

    # HC1 robust SEs (weighted)
    XtWX_inv = np.linalg.inv(Xw.T @ Xw)
    meat = np.zeros((k, k))
    for i in range(n):
        xi = X[i:i+1, :]
        meat += (w[i] * resid[i]) ** 2 * (xi.T @ xi)
    meat *= n / (n - k)
    V = XtWX_inv @ meat @ XtWX_inv
    se = np.sqrt(np.diag(V))
    t_stats = beta / se
    p_vals = 2 * stats.t.sf(np.abs(t_stats), df=n - k)
    ss_res = np.sum(w * resid ** 2)
    ss_tot = np.sum(w * (y - np.average(y, weights=w)) ** 2)
    r2 = 1 - ss_res / ss_tot
    if var_names is None:
        var_names = [f'x{i}' for i in range(k)]
    return {'beta': beta, 'se': se, 't': t_stats, 'p': p_vals,
            'r2': r2, 'n': n, 'names': var_names}

def print_reg(res, title=""):
    if title:
        print(f"\n{'='*80}")
        print(f"  {title}")
        print(f"{'='*80}")
    print(f"  N = {res['n']},  R² = {res['r2']:.4f}\n")
    print(f"  {'Variable':<45s} {'Coeff':>10s} {'SE':>10s} {'t':>8s} {'p':>8s}  ")
    print(f"  {'-'*45} {'-'*10} {'-'*10} {'-'*8} {'-'*8}--")
    for i, name in enumerate(res['names']):
        sig = "***" if res['p'][i]<.001 else "**" if res['p'][i]<.01 else "*" if res['p'][i]<.05 else "†" if res['p'][i]<.1 else ""
        print(f"  {name:<45s} {res['beta'][i]:>+10.4f} {res['se'][i]:>10.4f} {res['t'][i]:>8.3f} {res['p'][i]:>8.4f} {sig}")
    print()

# ── Load & prepare ───────────────────────────────────────────────────────────
df = pd.read_csv("tukwila.csv")
df = df[df['Free Reduced Lunch'] == 'Y'].copy()

df['abs_rate_2324'] = df['2023-2024 Absence Total'] / df['2023-2024 Member Days'] * 100
df['abs_rate_2425'] = df['2024-2025 Absence Total'] / df['2024-2025 Member Days'] * 100
df['cb_2324'] = (df['2023-2024 Chromebook'] == 'Y').astype(int)
df['cb_2425_only'] = ((df['2024-2025 Chromebook'] == 'Y') & (df['2023-2024 Chromebook'] == 'N')).astype(int)
df['ever_cb'] = ((df['cb_2324'] == 1) | (df['cb_2425_only'] == 1)).astype(int)
df['male'] = (df['Sex'] == 'M').astype(float)
df['sped'] = (df['Special Education'] == 'Y').astype(int)
df['ml'] = (df['2023-2024 Multilingual'] == 'Y').astype(int)


# ══════════════════════════════════════════════════════════════════════════════
# CEM IMPLEMENTATION
# ══════════════════════════════════════════════════════════════════════════════

def coarsen_exact_match(data, treatment_col, exact_cols, coarse_specs):
    """
    Coarsened Exact Matching.
    
    Parameters:
        data: DataFrame
        treatment_col: name of binary treatment column
        exact_cols: list of columns to match exactly
        coarse_specs: dict of {col: bins} for coarsened matching
            bins can be int (number of equal-width bins) or list of bin edges
    
    Returns:
        matched DataFrame with CEM weights
    """
    df_cem = data.copy()
    
    # Create coarsened versions
    strata_cols = list(exact_cols)
    for col, bins in coarse_specs.items():
        coarse_col = f'_cem_{col}'
        if isinstance(bins, int):
            df_cem[coarse_col] = pd.qcut(df_cem[col], q=bins, labels=False, duplicates='drop')
        else:
            df_cem[coarse_col] = pd.cut(df_cem[col], bins=bins, labels=False, include_lowest=True)
        strata_cols.append(coarse_col)
    
    # Create strata
    df_cem['_stratum'] = df_cem.groupby(strata_cols, dropna=False).ngroup()
    
    # Keep only strata with at least 1 treated AND 1 control
    strata_counts = df_cem.groupby('_stratum')[treatment_col].agg(['sum', 'count'])
    strata_counts.columns = ['n_treated', 'n_total']
    strata_counts['n_control'] = strata_counts['n_total'] - strata_counts['n_treated']
    valid_strata = strata_counts[(strata_counts['n_treated'] > 0) & (strata_counts['n_control'] > 0)].index
    
    matched = df_cem[df_cem['_stratum'].isin(valid_strata)].copy()
    
    # Compute CEM weights
    # Within each stratum, weight controls so they sum to the same as treated
    weights = np.ones(len(matched))
    for s in valid_strata:
        mask = matched['_stratum'] == s
        treated_mask = mask & (matched[treatment_col] == 1)
        control_mask = mask & (matched[treatment_col] == 0)
        n_t = treated_mask.sum()
        n_c = control_mask.sum()
        if n_c > 0 and n_t > 0:
            # Treated get weight 1, controls get weight n_t/n_c
            weights[control_mask.values] = n_t / n_c
    
    matched['_cem_weight'] = weights
    
    # Diagnostics
    n_strata_total = df_cem['_stratum'].nunique()
    n_strata_valid = len(valid_strata)
    n_treated_before = (data[treatment_col] == 1).sum()
    n_control_before = (data[treatment_col] == 0).sum()
    n_treated_after = (matched[treatment_col] == 1).sum()
    n_control_after = (matched[treatment_col] == 0).sum()
    
    diag = {
        'n_strata_total': n_strata_total,
        'n_strata_matched': n_strata_valid,
        'n_treated_before': n_treated_before,
        'n_treated_after': n_treated_after,
        'n_control_before': n_control_before,
        'n_control_after': n_control_after,
        'pct_treated_retained': 100 * n_treated_after / n_treated_before if n_treated_before > 0 else 0,
        'pct_control_retained': 100 * n_control_after / n_control_before if n_control_before > 0 else 0,
    }
    
    return matched, diag


def print_diagnostics(diag, title=""):
    if title:
        print(f"\n  {title}")
    print(f"  Total strata:       {diag['n_strata_total']}")
    print(f"  Matched strata:     {diag['n_strata_matched']}")
    print(f"  Treated: {diag['n_treated_before']} → {diag['n_treated_after']} ({diag['pct_treated_retained']:.1f}% retained)")
    print(f"  Control: {diag['n_control_before']} → {diag['n_control_after']} ({diag['pct_control_retained']:.1f}% retained)")


def balance_table(data, treatment_col, covariates, weights=None):
    """Print covariate balance before/after matching."""
    print(f"\n  {'Covariate':<30s} {'Treat Mean':>11s} {'Ctrl Mean':>11s} {'Diff':>8s} {'Std Diff':>9s}")
    print(f"  {'-'*30} {'-'*11} {'-'*11} {'-'*8} {'-'*9}")
    
    t_mask = data[treatment_col] == 1
    c_mask = data[treatment_col] == 0
    
    for cov in covariates:
        vals = data[cov].values.astype(float)
        if weights is not None:
            w = data[weights].values
            t_mean = np.average(vals[t_mask], weights=w[t_mask])
            c_mean = np.average(vals[c_mask], weights=w[c_mask])
        else:
            t_mean = vals[t_mask].mean()
            c_mean = vals[c_mask].mean()
        
        pooled_sd = vals.std()
        std_diff = (t_mean - c_mean) / pooled_sd if pooled_sd > 0 else 0
        
        flag = " ←" if abs(std_diff) > 0.1 else ""
        print(f"  {cov:<30s} {t_mean:>11.3f} {c_mean:>11.3f} {t_mean - c_mean:>+8.3f} {std_diff:>+9.3f}{flag}")


# ══════════════════════════════════════════════════════════════════════════════
# MATCHING SPEC 1: Ever CB → GPA
# Exact: sex, SPED, multilingual, race
# Coarse: GradYr (binned into elementary/middle/high)
# ══════════════════════════════════════════════════════════════════════════════

print("#" * 80)
print("# CEM SPECIFICATION 1: Ever Chromebook → GPA")
print("# Exact: sex, SPED, multilingual, race")
print("# Coarse: GradYr (3 bins: elementary/middle/high)")
print("#" * 80)

sub1 = df[['Cumulative GPA', 'ever_cb', 'male', 'sped', 'ml', 'LocalRace', 'GradYr']].dropna().copy()

# GradYr bins: elementary (2029-2031), middle (2027-2028), high (2024-2026)
gradyr_bins = [2023, 2026, 2028, 2032]

matched1, diag1 = coarsen_exact_match(
    sub1, 'ever_cb',
    exact_cols=['male', 'sped', 'ml', 'LocalRace'],
    coarse_specs={'GradYr': gradyr_bins}
)
print_diagnostics(diag1, "Matching Diagnostics:")

# Balance check
print("\n  BALANCE — BEFORE MATCHING (unweighted):")
balance_table(sub1, 'ever_cb', ['male', 'sped', 'ml', 'GradYr'])

print("\n  BALANCE — AFTER MATCHING (CEM weighted):")
balance_table(matched1, 'ever_cb', ['male', 'sped', 'ml', 'GradYr'], weights='_cem_weight')

# Simple weighted difference in means (SATT)
t_gpa = matched1.loc[matched1['ever_cb']==1, 'Cumulative GPA']
c_gpa = matched1.loc[matched1['ever_cb']==0, 'Cumulative GPA']
c_weights = matched1.loc[matched1['ever_cb']==0, '_cem_weight']

satt = t_gpa.mean() - np.average(c_gpa, weights=c_weights)
# SE via weighted t-test approximation
se_t = t_gpa.std(ddof=1) / np.sqrt(len(t_gpa))
se_c = np.sqrt(np.average((c_gpa - np.average(c_gpa, weights=c_weights))**2, weights=c_weights) / c_weights.sum())
se_satt = np.sqrt(se_t**2 + se_c**2)
t_satt = satt / se_satt
p_satt = 2 * stats.t.sf(abs(t_satt), df=len(t_gpa) + len(c_gpa) - 2)

print(f"\n  --- SATT (Sample Average Treatment Effect on the Treated) ---")
print(f"  Treated mean GPA:          {t_gpa.mean():.4f}")
print(f"  Weighted control mean GPA: {np.average(c_gpa, weights=c_weights):.4f}")
print(f"  SATT:  {satt:+.4f}")
print(f"  SE:    {se_satt:.4f}")
print(f"  t = {t_satt:.4f},  p = {p_satt:.4f}")
sig1 = "***" if p_satt<.001 else "**" if p_satt<.01 else "*" if p_satt<.05 else "†" if p_satt<.1 else "n.s."
print(f"  → {sig1}")

# WLS regression on matched sample (adds precision)
race_dum = pd.get_dummies(matched1['LocalRace'], prefix='race', drop_first=True, dtype=int)
matched1 = pd.concat([matched1, race_dum], axis=1)
race_cols = list(race_dum.columns)

var_wls = ['const', 'Ever Chromebook', 'male', 'sped', 'ml', 'GradYr'] + race_cols
X_wls = np.column_stack([
    np.ones(len(matched1)),
    matched1['ever_cb'].values,
    matched1['male'].values,
    matched1['sped'].values,
    matched1['ml'].values,
    matched1['GradYr'].values,
] + [matched1[c].values for c in race_cols])

res_wls1 = ols_with_se(X_wls, matched1['Cumulative GPA'].values, var_wls,
                        weights=matched1['_cem_weight'].values)
print_reg(res_wls1, "WLS on Matched Sample: GPA ~ Ever CB + Controls")


# ══════════════════════════════════════════════════════════════════════════════
# MATCHING SPEC 2: Ever CB → Absence Rate 2024-25
# Same matching variables + coarse on prior absence
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "#" * 80)
print("# CEM SPECIFICATION 2: Ever Chromebook → Absence Rate 2024-25")
print("# Exact: sex, SPED, multilingual, race")
print("# Coarse: GradYr (3 bins), Prior Absence Rate 2023-24 (quartiles)")
print("#" * 80)

sub2 = df[['abs_rate_2425', 'abs_rate_2324', 'ever_cb', 'male', 'sped', 'ml', 
           'LocalRace', 'GradYr']].dropna().copy()

# Prior absence in quartiles
abs_bins = [sub2['abs_rate_2324'].min() - 1,
            sub2['abs_rate_2324'].quantile(0.25),
            sub2['abs_rate_2324'].quantile(0.50),
            sub2['abs_rate_2324'].quantile(0.75),
            sub2['abs_rate_2324'].max() + 1]

matched2, diag2 = coarsen_exact_match(
    sub2, 'ever_cb',
    exact_cols=['male', 'sped', 'ml', 'LocalRace'],
    coarse_specs={'GradYr': gradyr_bins, 'abs_rate_2324': abs_bins}
)
print_diagnostics(diag2, "Matching Diagnostics:")

print("\n  BALANCE — BEFORE MATCHING:")
balance_table(sub2, 'ever_cb', ['male', 'sped', 'ml', 'GradYr', 'abs_rate_2324'])

print("\n  BALANCE — AFTER MATCHING (CEM weighted):")
balance_table(matched2, 'ever_cb', ['male', 'sped', 'ml', 'GradYr', 'abs_rate_2324'], weights='_cem_weight')

# SATT for absence rate
t_abs = matched2.loc[matched2['ever_cb']==1, 'abs_rate_2425']
c_abs = matched2.loc[matched2['ever_cb']==0, 'abs_rate_2425']
c_w2 = matched2.loc[matched2['ever_cb']==0, '_cem_weight']

satt2 = t_abs.mean() - np.average(c_abs, weights=c_w2)
se_t2 = t_abs.std(ddof=1) / np.sqrt(len(t_abs))
se_c2 = np.sqrt(np.average((c_abs - np.average(c_abs, weights=c_w2))**2, weights=c_w2) / c_w2.sum())
se_satt2 = np.sqrt(se_t2**2 + se_c2**2)
t_satt2 = satt2 / se_satt2
p_satt2 = 2 * stats.t.sf(abs(t_satt2), df=len(t_abs) + len(c_abs) - 2)

print(f"\n  --- SATT: Absence Rate 2024-25 ---")
print(f"  Treated mean:          {t_abs.mean():.2f}%")
print(f"  Weighted control mean: {np.average(c_abs, weights=c_w2):.2f}%")
print(f"  SATT:  {satt2:+.2f} pp")
print(f"  SE:    {se_satt2:.2f}")
print(f"  t = {t_satt2:.4f},  p = {p_satt2:.4f}")
sig2 = "***" if p_satt2<.001 else "**" if p_satt2<.01 else "*" if p_satt2<.05 else "†" if p_satt2<.1 else "n.s."
print(f"  → {sig2}")

# WLS
race_dum2 = pd.get_dummies(matched2['LocalRace'], prefix='race', drop_first=True, dtype=int)
matched2 = pd.concat([matched2, race_dum2], axis=1)
race_cols2 = list(race_dum2.columns)

var_wls2 = ['const', 'Ever Chromebook', 'Prior Absence 23-24', 'male', 'sped', 'ml', 'GradYr'] + race_cols2
X_wls2 = np.column_stack([
    np.ones(len(matched2)),
    matched2['ever_cb'].values,
    matched2['abs_rate_2324'].values,
    matched2['male'].values,
    matched2['sped'].values,
    matched2['ml'].values,
    matched2['GradYr'].values,
] + [matched2[c].values for c in race_cols2])

res_wls2 = ols_with_se(X_wls2, matched2['abs_rate_2425'].values, var_wls2,
                        weights=matched2['_cem_weight'].values)
print_reg(res_wls2, "WLS on Matched Sample: Absence Rate 24-25 ~ Ever CB + Prior Absence + Controls")


# ══════════════════════════════════════════════════════════════════════════════
# MATCHING SPEC 3: Separate cohorts — CB 2023-24 vs never, CB 2024-25 vs never
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "#" * 80)
print("# CEM SPECIFICATION 3: Separate cohort effects on GPA")
print("#" * 80)

for cb_label, cb_col in [('CB 2023-24', 'cb_2324'), ('CB 2024-25 only', 'cb_2425_only')]:
    sub3 = df[['Cumulative GPA', cb_col, 'male', 'sped', 'ml', 'LocalRace', 'GradYr']].dropna().copy()
    # Exclude the OTHER cb cohort from controls
    if cb_col == 'cb_2324':
        sub3 = sub3[~((sub3.index.isin(df[df['cb_2425_only']==1].index)) & (sub3[cb_col]==0))]
    else:
        sub3 = sub3[~((sub3.index.isin(df[df['cb_2324']==1].index)) & (sub3[cb_col]==0))]
    
    matched3, diag3 = coarsen_exact_match(
        sub3, cb_col,
        exact_cols=['male', 'sped', 'ml', 'LocalRace'],
        coarse_specs={'GradYr': gradyr_bins}
    )
    print_diagnostics(diag3, f"\n  {cb_label} — Matching Diagnostics:")
    
    t3 = matched3.loc[matched3[cb_col]==1, 'Cumulative GPA']
    c3 = matched3.loc[matched3[cb_col]==0, 'Cumulative GPA']
    w3 = matched3.loc[matched3[cb_col]==0, '_cem_weight']
    
    satt3 = t3.mean() - np.average(c3, weights=w3)
    se_t3 = t3.std(ddof=1) / np.sqrt(len(t3))
    se_c3 = np.sqrt(np.average((c3 - np.average(c3, weights=w3))**2, weights=w3) / w3.sum())
    se3 = np.sqrt(se_t3**2 + se_c3**2)
    t_stat3 = satt3 / se3
    p3 = 2 * stats.t.sf(abs(t_stat3), df=len(t3)+len(c3)-2)
    
    sig3 = "***" if p3<.001 else "**" if p3<.01 else "*" if p3<.05 else "†" if p3<.1 else "n.s."
    print(f"\n  {cb_label} SATT on GPA: {satt3:+.4f}  (SE={se3:.4f}, t={t_stat3:.4f}, p={p3:.4f}) {sig3}")
    print(f"    Treated mean: {t3.mean():.3f}  |  Weighted control mean: {np.average(c3, weights=w3):.3f}")


# ══════════════════════════════════════════════════════════════════════════════
# MATCHING SPEC 4: Staggered — CB 2023-24 (early) vs CB 2024-25 (late)
# Most credible comparison: both groups are CB-eligible
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "#" * 80)
print("# CEM SPECIFICATION 4: Early CB (2023-24) vs Late CB (2024-25 only)")
print("# Most credible: both groups are disadvantaged enough to qualify")
print("#" * 80)

sub4 = df[(df['cb_2324']==1) | (df['cb_2425_only']==1)].copy()
sub4['early'] = sub4['cb_2324']
sub4 = sub4[['Cumulative GPA', 'early', 'male', 'sped', 'ml', 'LocalRace', 'GradYr']].dropna()

matched4, diag4 = coarsen_exact_match(
    sub4, 'early',
    exact_cols=['male', 'sped', 'ml', 'LocalRace'],
    coarse_specs={'GradYr': gradyr_bins}
)
print_diagnostics(diag4, "Matching Diagnostics:")

print("\n  BALANCE — BEFORE:")
balance_table(sub4, 'early', ['male', 'sped', 'ml', 'GradYr'])
print("\n  BALANCE — AFTER (CEM weighted):")
balance_table(matched4, 'early', ['male', 'sped', 'ml', 'GradYr'], weights='_cem_weight')

t4 = matched4.loc[matched4['early']==1, 'Cumulative GPA']
c4 = matched4.loc[matched4['early']==0, 'Cumulative GPA']
w4 = matched4.loc[matched4['early']==0, '_cem_weight']

satt4 = t4.mean() - np.average(c4, weights=w4)
se_t4 = t4.std(ddof=1) / np.sqrt(len(t4))
se_c4 = np.sqrt(np.average((c4 - np.average(c4, weights=w4))**2, weights=w4) / w4.sum())
se4 = np.sqrt(se_t4**2 + se_c4**2)
t_stat4 = satt4 / se4
p4 = 2 * stats.t.sf(abs(t_stat4), df=len(t4)+len(c4)-2)

sig4 = "***" if p4<.001 else "**" if p4<.01 else "*" if p4<.05 else "†" if p4<.1 else "n.s."
print(f"\n  --- SATT: Early CB vs Late CB on GPA ---")
print(f"  Early CB mean GPA:   {t4.mean():.4f}")
print(f"  Late CB mean GPA:    {np.average(c4, weights=w4):.4f}")
print(f"  SATT:  {satt4:+.4f}")
print(f"  SE:    {se4:.4f}")
print(f"  t = {t_stat4:.4f},  p = {p4:.4f}")
print(f"  → {sig4}")


# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 80)
print("SUMMARY: Coarsened Exact Matching Results")
print("=" * 80)

print(f"""
  {'Specification':<55s} {'SATT':>8s} {'SE':>8s} {'p':>8s}
  {'-'*55} {'-'*8} {'-'*8} {'-'*8}
  1: Ever CB → GPA                                      {satt:>+8.3f} {se_satt:>8.3f} {p_satt:>8.4f} {sig1}
  1: Ever CB → GPA (WLS w/ controls)                    {res_wls1['beta'][1]:>+8.3f} {res_wls1['se'][1]:>8.3f} {res_wls1['p'][1]:>8.4f} {"***" if res_wls1['p'][1]<.001 else "**" if res_wls1['p'][1]<.01 else "*" if res_wls1['p'][1]<.05 else "†" if res_wls1['p'][1]<.1 else "n.s."}
  2: Ever CB → Absence Rate 24-25 (matched on prior)    {satt2:>+8.2f} {se_satt2:>8.2f} {p_satt2:>8.4f} {sig2}
  2: Ever CB → Abs Rate (WLS w/ controls)               {res_wls2['beta'][1]:>+8.2f} {res_wls2['se'][1]:>8.2f} {res_wls2['p'][1]:>8.4f} {"***" if res_wls2['p'][1]<.001 else "**" if res_wls2['p'][1]<.01 else "*" if res_wls2['p'][1]<.05 else "†" if res_wls2['p'][1]<.1 else "n.s."}
  4: Early CB vs Late CB → GPA                          {satt4:>+8.3f} {se4:>8.3f} {p4:>8.4f} {sig4}

NOTES:
  - SATT = Sample Average Treatment Effect on the Treated
  - Exact match: sex, SPED, multilingual, race
  - Coarse match: GradYr (elem/middle/high), prior absence (quartiles in Spec 2)
  - WLS models add regression adjustment on top of matching (doubly robust flavor)
  - All restricted to free/reduced lunch students
  - CEM weights ensure control group mirrors treated within each stratum
""")

################################################################################
# CEM SPECIFICATION 1: Ever Chromebook → GPA
# Exact: sex, SPED, multilingual, race
# Coarse: GradYr (3 bins: elementary/middle/high)
################################################################################

  Matching Diagnostics:
  Total strata:       123
  Matched strata:     56
  Treated: 226 → 222 (98.2% retained)
  Control: 1479 → 1103 (74.6% retained)

  BALANCE — BEFORE MATCHING (unweighted):

  Covariate                       Treat Mean   Ctrl Mean     Diff  Std Diff
  ------------------------------ ----------- ----------- -------- ---------
  male                                 0.473       0.492   -0.019    -0.038
  sped                                 0.066       0.051   +0.015    +0.067
  ml                                   0.770       0.380   +0.390    +0.787 ←
  GradYr                            2027.035    2027.606   -0.571    -0.256 ←

  BALANCE — AFTER MATCHING (CEM weighted):

 

In [9]:
"""
Inverse Probability Weighting (IPW): Chromebook Effect on GPA and Absence Rate
================================================================================

Steps:
  1. Estimate propensity score: P(Chromebook | covariates) via logistic regression
  2. Compute IPW weights: treated get 1/p, controls get 1/(1-p)
  3. Check diagnostics: weight distribution, covariate balance after weighting
  4. Estimate treatment effects using weighted regression
  5. Sensitivity: trimmed weights, stabilized weights, doubly robust

All restricted to free/reduced lunch students.
"""

import pandas as pd
import numpy as np
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")


# ── Logistic regression from scratch ─────────────────────────────────────────
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def logistic_fit(X, y):
    """Fit logistic regression via MLE. Returns coefficients."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, k = X.shape
    
    def neg_log_lik(beta):
        p = sigmoid(X @ beta)
        p = np.clip(p, 1e-10, 1 - 1e-10)
        return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    
    def gradient(beta):
        p = sigmoid(X @ beta)
        return -X.T @ (y - p)
    
    beta0 = np.zeros(k)
    result = minimize(neg_log_lik, beta0, jac=gradient, method='BFGS', 
                      options={'maxiter': 1000})
    return result.x

def logistic_predict(X, beta):
    return sigmoid(np.asarray(X, dtype=float) @ beta)


# ── WLS with robust SEs ─────────────────────────────────────────────────────
def wls_robust(X, y, weights, var_names=None):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(weights, dtype=float)
    n, k = X.shape
    
    W_sqrt = np.sqrt(w)
    Xw = X * W_sqrt[:, None]
    yw = y * W_sqrt
    
    beta = np.linalg.lstsq(Xw, yw, rcond=None)[0]
    resid = y - X @ beta
    
    # Sandwich (HC1) SEs
    XtWX_inv = np.linalg.inv(Xw.T @ Xw)
    meat = np.zeros((k, k))
    for i in range(n):
        xi = X[i:i+1, :]
        meat += (w[i] * resid[i]) ** 2 * (xi.T @ xi)
    meat *= n / (n - k)
    V = XtWX_inv @ meat @ XtWX_inv
    se = np.sqrt(np.diag(V))
    t_stats = beta / se
    p_vals = 2 * stats.t.sf(np.abs(t_stats), df=n - k)
    ss_res = np.sum(w * resid ** 2)
    ss_tot = np.sum(w * (y - np.average(y, weights=w)) ** 2)
    r2 = 1 - ss_res / ss_tot
    
    if var_names is None:
        var_names = [f'x{i}' for i in range(k)]
    return {'beta': beta, 'se': se, 't': t_stats, 'p': p_vals,
            'r2': r2, 'n': n, 'names': var_names, 'resid': resid}

def print_reg(res, title=""):
    if title:
        print(f"\n{'='*80}")
        print(f"  {title}")
        print(f"{'='*80}")
    print(f"  N = {res['n']},  R² = {res['r2']:.4f}\n")
    print(f"  {'Variable':<45s} {'Coeff':>10s} {'SE':>10s} {'t':>8s} {'p':>8s}  ")
    print(f"  {'-'*45} {'-'*10} {'-'*10} {'-'*8} {'-'*8}--")
    for i, name in enumerate(res['names']):
        sig = "***" if res['p'][i]<.001 else "**" if res['p'][i]<.01 else "*" if res['p'][i]<.05 else "†" if res['p'][i]<.1 else ""
        print(f"  {name:<45s} {res['beta'][i]:>+10.4f} {res['se'][i]:>10.4f} {res['t'][i]:>8.3f} {res['p'][i]:>8.4f} {sig}")
    print()


# ── Load & prepare ───────────────────────────────────────────────────────────
df = pd.read_csv("tukwila.csv")
df = df[df['Free Reduced Lunch'] == 'Y'].copy()

df['abs_rate_2324'] = df['2023-2024 Absence Total'] / df['2023-2024 Member Days'] * 100
df['abs_rate_2425'] = df['2024-2025 Absence Total'] / df['2024-2025 Member Days'] * 100
df['cb_2324'] = (df['2023-2024 Chromebook'] == 'Y').astype(int)
df['cb_2425_only'] = ((df['2024-2025 Chromebook'] == 'Y') & (df['2023-2024 Chromebook'] == 'N')).astype(int)
df['ever_cb'] = ((df['cb_2324'] == 1) | (df['cb_2425_only'] == 1)).astype(int)
df['male'] = (df['Sex'] == 'M').astype(float)
df['sped'] = (df['Special Education'] == 'Y').astype(int)
df['ml'] = (df['2023-2024 Multilingual'] == 'Y').astype(int)

race_dummies = pd.get_dummies(df['LocalRace'], prefix='race', drop_first=True, dtype=int)
df = pd.concat([df, race_dummies], axis=1)
race_cols = list(race_dummies.columns)

ps_covariates = ['male', 'sped', 'ml', 'GradYr'] + race_cols


# ══════════════════════════════════════════════════════════════════════════════
# HELPER: Full IPW pipeline
# ══════════════════════════════════════════════════════════════════════════════

def run_ipw(data, treatment_col, outcome_col, ps_covs, label="",
            trim_pct=None, stabilized=False, extra_controls=None):
    """
    Run full IPW analysis: propensity score → weights → diagnostics → ATE/ATT.
    """
    all_cols = list(set([treatment_col, outcome_col] + ps_covs + (extra_controls or [])))
    sub = data[all_cols].dropna().copy()
    
    treat = sub[treatment_col].values
    y = sub[outcome_col].values
    
    # Step 1: Propensity score model
    X_ps = np.column_stack([np.ones(len(sub))] + [sub[c].values for c in ps_covs])
    beta_ps = logistic_fit(X_ps, treat)
    ps = logistic_predict(X_ps, beta_ps)
    ps = np.clip(ps, 0.001, 0.999)  # prevent extreme weights
    
    print(f"\n{'#'*80}")
    print(f"# IPW: {label}")
    print(f"{'#'*80}")
    
    # Propensity score diagnostics
    print(f"\n  PROPENSITY SCORE MODEL")
    print(f"  N = {len(sub)} (Treated={treat.sum():.0f}, Control={(1-treat).sum():.0f})")
    print(f"\n  Propensity Score Distribution:")
    print(f"    {'':20s} {'Treated':>10s} {'Control':>10s}")
    print(f"    {'Mean':20s} {ps[treat==1].mean():>10.4f} {ps[treat==0].mean():>10.4f}")
    print(f"    {'Median':20s} {np.median(ps[treat==1]):>10.4f} {np.median(ps[treat==0]):>10.4f}")
    print(f"    {'Min':20s} {ps[treat==1].min():>10.4f} {ps[treat==0].min():>10.4f}")
    print(f"    {'Max':20s} {ps[treat==1].max():>10.4f} {ps[treat==0].max():>10.4f}")
    print(f"    {'SD':20s} {ps[treat==1].std():>10.4f} {ps[treat==0].std():>10.4f}")
    
    # Overlap check
    overlap_min = max(ps[treat==1].min(), ps[treat==0].min())
    overlap_max = min(ps[treat==1].max(), ps[treat==0].max())
    in_overlap = ((ps >= overlap_min) & (ps <= overlap_max)).sum()
    print(f"\n    Common support region: [{overlap_min:.4f}, {overlap_max:.4f}]")
    print(f"    Observations in common support: {in_overlap}/{len(sub)} ({100*in_overlap/len(sub):.1f}%)")
    
    # Step 2: Compute weights
    # ATE weights: treated get 1/p, controls get 1/(1-p)
    w_ate = np.where(treat == 1, 1/ps, 1/(1-ps))
    
    # ATT weights: treated get 1, controls get p/(1-p)
    w_att = np.where(treat == 1, 1, ps/(1-ps))
    
    if stabilized:
        p_treat = treat.mean()
        w_ate = np.where(treat == 1, p_treat/ps, (1-p_treat)/(1-ps))
        w_att = np.where(treat == 1, 1, (p_treat/(1-p_treat)) * (ps/(1-ps)))
    
    # Trimming
    if trim_pct:
        lo = np.percentile(ps, trim_pct)
        hi = np.percentile(ps, 100 - trim_pct)
        trim_mask = (ps >= lo) & (ps <= hi)
        n_trimmed = (~trim_mask).sum()
        sub = sub[trim_mask].copy()
        treat = treat[trim_mask]
        y = y[trim_mask]
        ps = ps[trim_mask]
        w_ate = w_ate[trim_mask]
        w_att = w_att[trim_mask]
        print(f"\n  TRIMMING: removed {n_trimmed} obs with PS outside [{lo:.4f}, {hi:.4f}]")
        print(f"  Remaining: {len(sub)} obs")
    
    # Weight diagnostics
    print(f"\n  WEIGHT DIAGNOSTICS {'(stabilized)' if stabilized else '(unstabilized)'}:")
    for w_label, w in [('ATE weights', w_ate), ('ATT weights', w_att)]:
        print(f"    {w_label}:")
        print(f"      Treated  — mean={w[treat==1].mean():.3f}, max={w[treat==1].max():.3f}, min={w[treat==1].min():.3f}")
        print(f"      Control  — mean={w[treat==0].mean():.3f}, max={w[treat==0].max():.3f}, min={w[treat==0].min():.3f}")
        ess_t = (w[treat==1].sum())**2 / (w[treat==1]**2).sum()
        ess_c = (w[treat==0].sum())**2 / (w[treat==0]**2).sum()
        print(f"      ESS: treated={ess_t:.1f}, control={ess_c:.1f}")
    
    # Step 3: Covariate balance after weighting
    print(f"\n  COVARIATE BALANCE (ATT weights):")
    print(f"  {'Covariate':<30s} {'Unwtd Diff':>10s} {'Wtd Diff':>10s} {'Unwtd SD':>9s} {'Wtd SD':>9s}")
    print(f"  {'-'*30} {'-'*10} {'-'*10} {'-'*9} {'-'*9}")
    
    for cov in ps_covs:
        vals = sub[cov].values.astype(float)
        pooled_sd = vals.std()
        if pooled_sd == 0:
            continue
        
        # Unweighted
        diff_raw = vals[treat==1].mean() - vals[treat==0].mean()
        sd_raw = diff_raw / pooled_sd
        
        # ATT weighted
        t_mean = np.average(vals[treat==1], weights=w_att[treat==1])
        c_mean = np.average(vals[treat==0], weights=w_att[treat==0])
        diff_wtd = t_mean - c_mean
        sd_wtd = diff_wtd / pooled_sd
        
        flag = " ←" if abs(sd_wtd) > 0.1 else ""
        print(f"  {cov:<30s} {diff_raw:>+10.3f} {diff_wtd:>+10.3f} {sd_raw:>+9.3f} {sd_wtd:>+9.3f}{flag}")
    
    # Step 4: Estimate treatment effects
    results = {}
    
    # --- ATE: weighted difference in means ---
    ate_t = np.average(y[treat==1], weights=w_ate[treat==1])
    ate_c = np.average(y[treat==0], weights=w_ate[treat==0])
    ate = ate_t - ate_c
    
    # --- ATT: weighted difference ---
    att_t = y[treat==1].mean()  # unweighted for treated
    att_c = np.average(y[treat==0], weights=w_att[treat==0])
    att = att_t - att_c
    
    # --- ATE via WLS regression ---
    X_ate = np.column_stack([np.ones(len(sub)), treat])
    res_ate = wls_robust(X_ate, y, w_ate, ['const', f'{treatment_col} (ATE)'])
    
    # --- ATT via WLS regression ---
    res_att = wls_robust(X_ate, y, w_att, ['const', f'{treatment_col} (ATT)'])
    
    print(f"\n  --- TREATMENT EFFECTS ---")
    print(f"  ATE  (avg effect on everyone):     {ate:>+.4f}")
    print(f"  ATT  (avg effect on treated):      {att:>+.4f}")
    
    print_reg(res_ate, f"ATE via WLS: {outcome_col} ~ {treatment_col}")
    print_reg(res_att, f"ATT via WLS: {outcome_col} ~ {treatment_col}")
    
    # Step 5: Doubly robust — WLS with controls on IPW-weighted sample
    if extra_controls:
        ctrl_list = extra_controls
    else:
        ctrl_list = ps_covs
    
    dr_vars = ['const', f'{treatment_col}'] + ctrl_list
    X_dr = np.column_stack([
        np.ones(len(sub)),
        treat,
    ] + [sub[c].values for c in ctrl_list])
    
    res_dr_ate = wls_robust(X_dr, y, w_ate, dr_vars)
    res_dr_att = wls_robust(X_dr, y, w_att, dr_vars)
    
    print_reg(res_dr_ate, f"Doubly Robust (ATE): {outcome_col} ~ {treatment_col} + controls, IPW-weighted")
    print_reg(res_dr_att, f"Doubly Robust (ATT): {outcome_col} ~ {treatment_col} + controls, IPW-weighted")
    
    results['ate_simple'] = ate
    results['att_simple'] = att
    results['ate_wls'] = (res_ate['beta'][1], res_ate['se'][1], res_ate['p'][1])
    results['att_wls'] = (res_att['beta'][1], res_att['se'][1], res_att['p'][1])
    results['dr_ate'] = (res_dr_ate['beta'][1], res_dr_ate['se'][1], res_dr_ate['p'][1])
    results['dr_att'] = (res_dr_att['beta'][1], res_dr_att['se'][1], res_dr_att['p'][1])
    
    return results


# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 1: Ever CB → GPA
# ══════════════════════════════════════════════════════════════════════════════

r1 = run_ipw(df, 'ever_cb', 'Cumulative GPA', ps_covariates,
             label="Ever Chromebook → Cumulative GPA")

# With stabilized weights
r1s = run_ipw(df, 'ever_cb', 'Cumulative GPA', ps_covariates,
              label="Ever Chromebook → GPA (STABILIZED weights)", stabilized=True)

# With trimming
r1t = run_ipw(df, 'ever_cb', 'Cumulative GPA', ps_covariates,
              label="Ever Chromebook → GPA (TRIMMED 5th/95th)", trim_pct=5)


# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 2: Ever CB → Absence Rate 2024-25 (controlling for prior absence)
# ══════════════════════════════════════════════════════════════════════════════

r2 = run_ipw(df, 'ever_cb', 'abs_rate_2425', ps_covariates,
             label="Ever Chromebook → Absence Rate 2024-25",
             extra_controls=ps_covariates + ['abs_rate_2324'])


# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 3: Separate cohorts on GPA
# ══════════════════════════════════════════════════════════════════════════════

# CB 2023-24 vs never CB
df_no2425 = df[df['cb_2425_only'] == 0].copy()
r3a = run_ipw(df_no2425, 'cb_2324', 'Cumulative GPA', ps_covariates,
              label="CB 2023-24 vs Never CB → GPA")

# CB 2024-25 vs never CB
df_no2324 = df[df['cb_2324'] == 0].copy()
r3b = run_ipw(df_no2324, 'cb_2425_only', 'Cumulative GPA', ps_covariates,
              label="CB 2024-25 only vs Never CB → GPA")


# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 4: CB 2023-24 Absence Rate 2023-24 (contemporaneous)
# ══════════════════════════════════════════════════════════════════════════════

r4 = run_ipw(df_no2425, 'cb_2324', 'abs_rate_2324', ps_covariates,
             label="CB 2023-24 → Absence Rate 2023-24 (contemporaneous)")


# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("SUMMARY: IPW Treatment Effect Estimates")
print("=" * 80)

def fmt_result(r, key):
    b, se, p = r[key]
    sig = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else "†" if p<.1 else ""
    return f"{b:>+8.3f} ({se:.3f}) p={p:.4f} {sig}"

print(f"""
  {'Analysis':<50s} {'ATT (WLS)':>35s}
  {'-'*50} {'-'*35}
  Ever CB → GPA                                    {fmt_result(r1, 'att_wls')}
  Ever CB → GPA (stabilized)                       {fmt_result(r1s, 'att_wls')}
  Ever CB → GPA (trimmed)                          {fmt_result(r1t, 'att_wls')}
  CB 2023-24 → GPA                                {fmt_result(r3a, 'att_wls')}
  CB 2024-25 → GPA                                {fmt_result(r3b, 'att_wls')}
  
  {'Analysis':<50s} {'DR-ATT':>35s}
  {'-'*50} {'-'*35}
  Ever CB → GPA (doubly robust)                    {fmt_result(r1, 'dr_att')}
  Ever CB → Absence 24-25 (DR, w/ prior abs)       {fmt_result(r2, 'dr_att')}
  CB 2023-24 → Absence 23-24 (DR, contemporaneous) {fmt_result(r4, 'dr_att')}

NOTES:
  - ATE = Average Treatment Effect (on entire population)
  - ATT = Average Treatment Effect on the Treated (most relevant here)
  - DR = Doubly Robust (IPW + regression controls; consistent if EITHER the
    propensity model OR the outcome model is correctly specified)
  - Stabilized weights multiply by P(treatment) to reduce variance
  - Trimming removes extreme propensity scores (5th/95th percentile)
  - All restricted to free/reduced lunch students
  - HC1 robust sandwich SEs throughout
""")


################################################################################
# IPW: Ever Chromebook → Cumulative GPA
################################################################################

  PROPENSITY SCORE MODEL
  N = 1705 (Treated=226, Control=1479)

  Propensity Score Distribution:
                            Treated    Control
    Mean                     0.2205     0.1191
    Median                   0.2591     0.0731
    Min                      0.0237     0.0111
    Max                      0.5482     0.6249
    SD                       0.1091     0.0954

    Common support region: [0.0237, 0.5482]
    Observations in common support: 1635/1705 (95.9%)

  WEIGHT DIAGNOSTICS (unstabilized):
    ATE weights:
      Treated  — mean=8.486, max=42.154, min=1.824
      Control  — mean=1.151, max=2.666, min=1.011
      ESS: treated=102.8, control=1455.0
    ATT weights:
      Treated  — mean=1.000, max=1.000, min=1.000
      Control  — mean=0.151, max=1.666, min=0.011
   